# Fase 2 - Primer contacto con PySpark

## Sesión

In [1]:
import os, time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = (
    SparkSession.builder
    .appName("fase2-sitio3")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext
time.sleep(3)
print("Spark      :", spark.version)
print("Master     :", sc.master)
print("Executors  :", sc._jsc.sc().getExecutorMemoryStatus().size() - 1)
print("Paralelismo:", sc.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/29 21:49:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark      : 4.0.1
Master     : spark://spark-master:7077
Executors  : 2
Paralelismo: 4


## Evaluación perezosa

In [2]:
RAW = "/opt/workspace/data/raw/*.jsonl"
esquema = StructType([
    StructField("timestamp",  StringType(), True),
    StructField("sensor_id",  StringType(), True),
    StructField("metric",     StringType(), True),
    StructField("value",      DoubleType(), True),
    StructField("unit",       StringType(), True),
    StructField("worker_id",  LongType(),   True),
    StructField("site",       StringType(), True),
    StructField("asset_type", StringType(), True),
    StructField("status",     StringType(), True),
    StructField("seq",        LongType(),   True),
])

t0 = time.perf_counter()
df = spark.read.schema(esquema).json(RAW)      # no lee nada todavía
limpio = df.filter(F.col("value") > -18)       # no filtra nada todavía
print(f"read + filter: {time.perf_counter() - t0:.2f} s")

t0 = time.perf_counter()
n = limpio.count()                             # aquí se ejecuta todo
print(f"count        : {time.perf_counter() - t0:.2f} s -> {n:,} filas")

read + filter: 0.68 s


count        : 9.24 s -> 8,770,606 filas


## Esquema inferido vs declarado

In [3]:
t0 = time.perf_counter()
df_inferido = spark.read.json(RAW)
n_inf = df_inferido.count()
t_inf = time.perf_counter() - t0

t0 = time.perf_counter()
df = spark.read.schema(esquema).json(RAW)
n_decl = df.count()
t_decl = time.perf_counter() - t0

print(f"inferido : {t_inf:.2f} s ({n_inf:,} filas)")
print(f"declarado: {t_decl:.2f} s ({n_decl:,} filas)")

inferido : 13.11 s (10,000,014 filas)
declarado: 7.43 s (10,000,014 filas)


In [4]:
df.printSchema()
df.show(5, truncate=False)

root
 |-- timestamp: string (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- worker_id: long (nullable = true)
 |-- site: string (nullable = true)
 |-- asset_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- seq: long (nullable = true)

+------------------------+------------+-----------+------+----+---------+----+----------+------+---+
|timestamp               |sensor_id   |metric     |value |unit|worker_id|site|asset_type|status|seq|
+------------------------+------------+-----------+------+----+---------+----+----------+------+---+
|2026-08-24T00:00:00.000Z|REEFER_S3_01|temperature|-17.88|C   |0        |S3  |reefer    |OK    |0  |
|2026-08-24T00:00:00.000Z|REEFER_S3_01|humidity   |84.1  |%   |0        |S3  |reefer    |OK    |0  |
|2026-08-24T00:00:00.000Z|REEFER_S3_01|current    |11.35 |A   |0        |S3  |reefer    |OK    |0  |
|2026-0

In [5]:
spark.stop()